# Gemini book embeddings setup

Prepare the database connection and inspect the book and embedding tables before implementing embedding ingestion. This notebook currently checks setup and schema; it does not generate or store embeddings.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from sqlalchemy import create_engine, text, inspect

SUPABASE_DATABASE_URL = os.getenv("SUPABASE_DATABASE_URL")
engine = create_engine(SUPABASE_DATABASE_URL)

## Check database connectivity

Run a lightweight query to confirm the database connection is available.

In [3]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("Connection successful:", result.scalar())

except Exception as e:
    print("Connection failed:", e)

Connection successful: 1


## Register the pgvector type

Teach SQLAlchemy to recognize PostgreSQL `vector` columns so their schema can be inspected correctly.

In [6]:
from pgvector.sqlalchemy import VECTOR
from sqlalchemy.dialects.postgresql.base import ischema_names

ischema_names["vector"] = VECTOR

## Inspect the destination tables

Review the columns, nullability, and foreign keys for `books` and `book_embeddings` before adding embedding records.

In [7]:
inspector = inspect(engine)

for table in ["books", "book_embeddings"]:
    print(f"\n{table.upper()}")

    for column in inspector.get_columns(table):
        print(
            column["name"],
            column["type"],
            "nullable:", column["nullable"]
        )

    print("Foreign keys:", inspector.get_foreign_keys(table))


BOOKS
id BIGINT nullable: False
isbn13 VARCHAR nullable: True
isbn10 VARCHAR nullable: True
title VARCHAR nullable: False
authors TEXT nullable: True
categories TEXT nullable: True
thumbnail TEXT nullable: True
description TEXT nullable: True
published_year INTEGER nullable: True
average_rating DOUBLE PRECISION nullable: True
num_pages INTEGER nullable: True
ratings_count INTEGER nullable: True
title_and_subtitle TEXT nullable: True
tagged_description TEXT nullable: True
simple_categories TEXT nullable: True
anger DOUBLE PRECISION nullable: True
disgust DOUBLE PRECISION nullable: True
fear DOUBLE PRECISION nullable: True
joy DOUBLE PRECISION nullable: True
sadness DOUBLE PRECISION nullable: True
surprise DOUBLE PRECISION nullable: True
neutral DOUBLE PRECISION nullable: True
Foreign keys: []

BOOK_EMBEDDINGS
id BIGINT nullable: False
book_id BIGINT nullable: False
model VARCHAR nullable: False
dimensions INTEGER nullable: False
embedding VECTOR(768) nullable: False
Foreign keys: [{'na